In [1]:
from feupy.naima.utils import  make_naima_tables

In [1]:
# Licensed under a 3-clause BSD style license - see LICENSE.rst
"""
Configuration system for Naima spectral modeling and MCMC analysis.
Gammapy-style, YAML-serializable, with flexible input tables and particle setup.
"""

import json
import yaml
import logging
from pathlib import Path
from typing import Union, List, Optional, Annotated
from collections import defaultdict

import astropy.units as u
from astropy.table import Table

from pydantic.functional_validators import BeforeValidator

from gammapy.analysis.config import GammapyBaseConfig, GeneralConfig, deep_update
from gammapy.utils.scripts import read_yaml, write_yaml, to_yaml
from gammapy.utils.types import EnergyType, QuantityType
from gammapy.datasets import Datasets

from feupy.utils.enum import ParticleTypeEnum
from feupy.utils.enum import NaimaFunctionalModelsEnum


log = logging.getLogger(__name__)

# ==========================================================================
# HELPERS FOR INPUT VALIDATION
# ==========================================================================

def _load_single_table(path: Path):
    """Load a Naima-compatible flux table from ECSV/FITS/CSV."""
    suffix = path.suffix.lower()

    if suffix in [".ecsv", ".fits", ".csv"]:
        return Table.read(path)

    raise ValueError(f"Unsupported spectral file: {path}")


def _load_datasets_yaml(path: Path):
    """Load Gammapy Datasets YAML."""
    return Datasets.read(filename=str(path))


def validate_data_table(value):
    """
    Flexible validator that accepts:
    - Table
    - list of Tables
    - path to ECSV/FITS/CSV
    - path to datasets.yaml
    - list of paths
    """
    if value is None:
        return None

    if isinstance(value, Table):
        return [value]

    if isinstance(value, list) and all(isinstance(v, Table) for v in value):
        return value

    # Accept strings
    if isinstance(value, str):
        value = [value]

    if isinstance(value, list) and all(isinstance(v, str) for v in value):
        loaded = []

        for v in value:
            path = Path(v)
            if not path.exists():
                raise FileNotFoundError(f"Data table not found: {v}")

            if path.name == "datasets.yaml":
                loaded.append(_load_datasets_yaml(path))
            else:
                loaded.append(_load_single_table(path))

        return loaded

    raise ValueError(
        "data_table must be Table, list[Table], path, or list[path]."
    )


# ==========================================================================
# PRIORS
# ==========================================================================

class NaimaPriorConfig(GammapyBaseConfig):
    type: str = "uniform"
    limits: Optional[List[float]] = None


# ==========================================================================
# SAMPLER SETTINGS
# ==========================================================================

class NaimaSamplerConfig(GammapyBaseConfig):
    nwalkers: int = 32
    nburn: int = 2000
    nrun: int = 300
    threads: int = 1
    prefit: bool = True
    interactive: bool = False


# ==========================================================================
# OUTPUT SETTINGS
# ==========================================================================

class NaimaOutputConfig(GammapyBaseConfig):
    root: str = "naima_output"
    overwrite: bool = True


# ==========================================================================
# PARTICLE ENERGY SETTINGS
# ==========================================================================

class ParticleEnergyConfig(GammapyBaseConfig):
    min: Optional[EnergyType] = None
    max: Optional[EnergyType] = None
    ref: Optional[EnergyType] = None


# ==========================================================================
# INVERSE COMPTON SEED FIELDS
# ==========================================================================

class ICSeedFieldConfig(GammapyBaseConfig):
    name: str = "CMB"
    temperature: QuantityType = 2.7 * u.K
    energy_density: QuantityType = 0.25 * u.eV / u.cm**3

    def to_naima(self):
        return (self.name, self.temperature, self.energy_density)


# ==========================================================================
# PION DECAY
# ==========================================================================

class PionDecayDensityConfig(GammapyBaseConfig):
    nh: QuantityType = 1.0 / u.cm**3


# ==========================================================================
# RUN CONFIG (INPUT TABLES)
# ==========================================================================

class NaimaRunConfig(GammapyBaseConfig):
    data_table: Annotated[
        Optional[List[Union[str, Table]]],
        BeforeValidator(validate_data_table)
    ] = None


# ==========================================================================
# MODEL CONFIG
# ==========================================================================

class NaimaModelConfig(GammapyBaseConfig):

    particle_type: ParticleTypeEnum = "electrons"
    distribution_type: NaimaFunctionalModelsEnum = "PowerLaw"

    # Energies
    electron_energy: ParticleEnergyConfig = ParticleEnergyConfig(
        min=1e9 * u.eV,
        max=1e15 * u.eV,
        ref=1e12 * u.eV,
    )

    proton_energy: ParticleEnergyConfig = ParticleEnergyConfig()

    seed_photon_fields: List[ICSeedFieldConfig] = [
        ICSeedFieldConfig(
            name="CMB",
            temperature=2.7 * u.K,
            energy_density=0.25 * u.eV / u.cm**3,
        ),
        ICSeedFieldConfig(
            name="FIR",
            temperature=40 * u.K,
            energy_density=1.0 * u.eV/ u.cm**3,
        ),
        ICSeedFieldConfig(
            name="NIR",
            temperature=500 * u.K,
            energy_density=0.4 * u.eV/ u.cm**3,
        ),
        ICSeedFieldConfig(
            name="VIS",
            temperature=3500 * u.K,
            energy_density=1.9 * u.eV/ u.cm**3,
        ),
    ]

    pion_decay: PionDecayDensityConfig = PionDecayDensityConfig()

    @property
    def ic_fields_naima(self):
        """Return IC fields in Naima format."""
        return [f.to_naima() for f in self.seed_photon_fields]

    def get_energy_config(self, particle: str):
        """Return correct energy configuration."""
        if particle == "electrons":
            return self.electron_energy
        if particle == "protons":
            return self.proton_energy
        if particle == "electrons+protons":
            return [self.electron_energy, self.proton_energy]
        raise ValueError(f"Invalid particle type: {particle}")


# ==========================================================================
# TOP LEVEL CONFIG
# ==========================================================================

class NaimaMCMCConfig(GammapyBaseConfig):

    general: GeneralConfig = GeneralConfig()
    run: NaimaRunConfig = NaimaRunConfig()
    model: NaimaModelConfig = NaimaModelConfig()
    sampler: NaimaSamplerConfig = NaimaSamplerConfig()
    output: NaimaOutputConfig = NaimaOutputConfig()

    def __str__(self):
        """Display settings in pretty YAML format."""
        info = self.__class__.__name__ + "\n\n\t"
        data = self.to_yaml()
        data = data.replace("\n", "\n\t")
        info += data
        return info.expandtabs(tabsize=4)

    @classmethod
    def read(cls, path):
        """Read from YAML file.

        Parameters
        ----------
        path : str
            input filepath
        """
        config = read_yaml(path)
        config.pop("metadata", None)
        return AnalysisConfig(**config)

    @classmethod
    def from_yaml(cls, config_str):
        """Create from YAML string.

        Parameters
        ----------
        config_str : str
            yaml str

        """
        settings = yaml.safe_load(config_str)
        return NaimaMCMCConfig(**settings)

    def write(self, path, overwrite=False):
        """Write to YAML file.

        Parameters
        ----------
        path : `pathlib.Path` or str
            Path to write files.
        overwrite : bool, optional
            Overwrite existing file. Default is False.
        """
        yaml_str = self.to_yaml()
        write_yaml(yaml_str, path, overwrite=overwrite)

    def to_yaml(self):
        """Convert to YAML string."""
        data = json.loads(self.model_dump_json())
        return to_yaml(data)

    def set_logging(self):
        """Set logging config.

        Calls ``logging.basicConfig``, i.e. adjusts global logging state.
        """
        self.general.log.level = self.general.log.level.upper()
        logging.basicConfig(**self.general.log.model_dump())
        log.info("Setting logging config: {!r}".format(self.general.log.model_dump()))

    def update(self, config=None):
        """Update config with provided settings.

        Parameters
        ----------
        config : str or `AnalysisConfig` object, optional
            Configuration settings provided in dict() syntax. Default is None.
        """
        if isinstance(config, str):
            other = NaimaMCMCConfig.from_yaml(config)
        elif isinstance(config, NaimaMCMCConfig):
            other = config
        else:
            raise TypeError(f"Invalid type: {config}")

        config_new = deep_update(
            self.model_dump(exclude_defaults=True),
            other.model_dump(exclude_defaults=True),
        )
        return NaimaMCMCConfig(**config_new)

    @staticmethod
    def _get_doc_sections():
        """Return dictionary with commented docs from docs file."""
        doc = defaultdict(str)
        with open(DOCS_FILE) as f:
            for line in filter(lambda line: not line.startswith("---"), f):
                line = line.strip("\n")
                if line.startswith("# Section: "):
                    keyword = line.replace("# Section: ", "")
                doc[keyword] += line + "\n"
        return doc

In [2]:
config = NaimaMCMCConfig()
print(config)


NaimaMCMCConfig

    general:
        log:
            level: info
            filename: null
            filemode: null
            format: null
            datefmt: null
        outdir: .
        n_jobs: 1
        datasets_file: null
        models_file: null
    run:
        data_table: null
    model:
        particle_type: electrons
        distribution_type: PowerLaw
        electron_energy:
            min: 1000000000.0 eV
            max: 1000000000000000.0 eV
            ref: 1000000000000.0 eV
        proton_energy:
            min: null
            max: null
            ref: null
        seed_photon_fields:
        -   name: CMB
            temperature: 2.7 K
            energy_density: 0.25 eV / cm3
        -   name: FIR
            temperature: 40.0 K
            energy_density: 1.0 eV / cm3
        -   name: NIR
            temperature: 500.0 K
            energy_density: 0.4 eV / cm3
        -   name: VIS
            temperature: 3500.0 K
            energy_density: 1.9 

In [3]:
import sys
from pathlib import Path

def module_path():
    if hasattr(sys, 'ps1') or 'ipykernel' in sys.modules:
        # Estamos em Notebook ou IPython
        return Path.cwd()
    else:
        # Estamos num script ou módulo
        return Path(__file__).resolve().parent

from gammapy.datasets import Datasets
from pathlib import Path

BASE_PATH = module_path()
DATA_PATH = BASE_PATH / "data"


datasets = Datasets.read(
    filename = DATA_PATH / "datasets.yaml",
    filename_models = DATA_PATH / "models.yaml"
)
data_table = make_naima_tables(datasets)

/home/phoenix/miniconda3/envs/feupy/lib/python3.12/site-packages/gammapy/utils/scripts.py:71: UserWarning: Checksum verification failed.
  warnings.warn("Checksum verification failed.", UserWarning)
INFO:gammapy.estimators.points.core:Inferred format: gadf-sed
INFO:gammapy.estimators.points.core:Inferred format: gadf-sed
INFO:gammapy.estimators.points.core:Inferred format: gadf-sed
INFO:gammapy.estimators.points.core:Inferred format: gadf-sed
INFO:gammapy.estimators.points.core:Inferred format: gadf-sed
/home/phoenix/miniconda3/envs/feupy/lib/python3.12/site-packages/gammapy/utils/scripts.py:71: UserWarning: Checksum verification failed.
  warnings.warn("Checksum verification failed.", UserWarning)


NameError: name 'make_naima_tables' is not defined

In [ ]:

from feupy.utils.table import write_tables_csv

for table in data_table:
    meta = table.meta

    write_tables_csv(table, DATA_PATH, meta["name"])
    
    print(meta)

In [ ]:
config = NaimaMCMCConfig()

config

In [18]:
from gammapy.utils.scripts import make_path


In [19]:
filename = DATA_PATH / "datasets.yaml"
filename

config.run.data_table = data_table

In [23]:
print(config)


PydanticSerializationError: Unable to serialize unknown type: <class 'astropy.table.table.Table'>

In [21]:
log = logging.getLogger(__name__)

class NaimaMCMCAnalysis:
    """
        Configuration options following `NaimaMCMCAnalysisConfig` schema
    """

    def __init__(self, config):
        self.config = config
        self.config.set_logging()

    def _repr_html_(self):
        try:
            return self.to_html()
        except AttributeError:
            return f"<pre>{html.escape(str(self))}</pre>"
            
    @property
    def models(self):
        if not self.datasets:
            raise RuntimeError("No datasets defined. Impossible to set models.")
        return self.datasets.models

    @models.setter
    def models(self, models):
        self.set_models(models, extend=False)
        
    @property
    def config(self):
        """Simulation configuration (`NaimaMCMCConfig`)"""
        return self._config

    @config.setter
    def config(self, value):
        if isinstance(value, dict):
            self._config = NaimaMCMCConfig(**value)
        elif isinstance(value, NaimaMCMCConfig):
            self._config = value
        else:
            raise TypeError("config must be dict or NaimaMCMCConfig.")

   

In [22]:
analysis = NaimaMCMCAnalysis(config)

INFO:__main__:Setting logging config: {'level': 'INFO', 'filename': None, 'filemode': None, 'format': None, 'datefmt': None}


In [ ]:
 def _get_catalogs(self):
        """Load source catalogs and filter sources within the ROI."""
        catalogs = []
        config_settings = self.config
        all_catalogs = load_catalogs()
        position = convert_pos_config_to_skycoord(config_settings.roi)
        radius = config_settings.roi.radius

        for catalog in all_catalogs:
            separation = position.separation(catalog.positions)
            mask_roi = separation < radius

            if len(catalog[mask_roi].table):
                catalogs.append(catalog[mask_roi])
        return catalogs

    def _get_sources(self):
        """Retrieve sources from catalogs within the ROI."""
        sources = []
        for catalog in self._get_catalogs():
            sources.extend(catalog)
        self.sources =  Sources(sources)

    def _get_catalog_roi(self):
        """
        Compute separations of sources from the target position and return an Astropy Table.

        Returns
        -------
        result_table : `~astropy.table.Table`
            Table with columns for source name, catalog, RA, Dec, and separation from target.
        """
        names, ras, decs, separations, catalogs, has_fp = [], [], [], [], [], []
        sources = self.sources
        config_settings = self.config
        target_position = convert_pos_config_to_skycoord(config_settings.roi)

        for source in sources:
            source_name = source.name
            source_pos = source.position
            ra, dec = source_pos.ra.deg, source_pos.dec.deg
            catalog = get_catalog_tag(source)
            sep = source_pos.separation(target_position).deg

            names.append(source_name)
            ras.append(ra)
            decs.append(dec)
            catalogs.append(catalog)
            separations.append(sep)
            has_fp.append(self._has_fp[source.name])
            
        result_table = Table()
        result_table['index'] =  [_ for _ in range(0, (len(names)))]
        result_table['source_name'] = names
        result_table['source_label'] = sources.labels
        result_table['catalog'] = catalogs
        result_table['ra'] = ras * u.deg
        result_table['dec'] = decs * u.deg
        result_table['separation'] = separations * u.deg
        result_table['has_fp'] = has_fp
        
        for column in result_table.colnames:
            if column.startswith(("ra", "dec", "sep")):
                result_table[column].format = ".3f"
                
        # Add meta (columns descriptions)
        result_table.meta['description'] = {
            'index': "Unique identifier for each source.",
            'source_name': "Name of the source as listed in the catalog.",
            'source_label': "User-defined label for the source, often used for plotting.",
            'has_fp': "Boolean flag indicating if flux points are available (True/False).",
            'catalog': "Catalog from which the source originates.",
            'ra': "Right Ascension (RA) of the source in degrees, formatted to 3 decimal places.",
            'dec': "Declination (Dec) of the source in degrees, formatted to 3 decimal places.",
            'separation': "Angular separation from a reference position in degrees, formatted to 3 decimal places."
        }
        self.catalog = result_table      


    def run(self):
        """Run the ROI analysis, initializing datasets."""
        self._get_sources()
        self._get_flux_points_datasets()
        self._get_catalog_roi()

    def _get_flux_points_datasets(self):
        """Generate FluxPointsDataset objects from sources within the ROI."""
        models = Models()
        datasets = Datasets()
        dict_fp = {}
        sources = self.sources
        log.info("Generating FluxPointsDataset.")

        for index, source in enumerate(sources):
            label = sources.labels[index]
            tag = get_catalog_tag(source)
            if tag == "psrcat":
                dict_fp[source.name] = False
                log.info("Flux points unavailable for ATNF Pulsar Catalog.")
                continue     
                
            try:
                which = None

                if tag == "2PC":
                    spectral_model = self._create_spectral_model(source, which)
                    flux_points = get_flux_points_2PC(source)
                    dict_fp[source.name] = True
                    self._create_flux_points_dataset(f"PSR {label}", models, datasets,  spectral_model, flux_points)

                elif tag == "3PC":
                    spectral_model = self._create_spectral_model(source, which)
                    flux_points = get_flux_points_3PC(source)
                    dict_fp[source.name] = True
                    # self._create_flux_points_dataset(f"PSR {label}", models, datasets,  spectral_model, flux_points)
                    self._create_flux_points_dataset(f"{label}", models, datasets,  spectral_model, flux_points)
                    
                elif tag == "3hwc":
                    spectral_model = self._create_spectral_model(source, which)
                    flux_points = get_flux_points_3hwc(source)
                    dict_fp[source.name] = True
                    # self._create_flux_points_dataset(label, models, datasets,  spectral_model, flux_points)
                    self._create_flux_points_dataset(f"{label}", models, datasets,  spectral_model, flux_points)

                elif tag == "2hwc":
                    for model_type in ['point', 'extended']:
                        spectral_model = self._create_spectral_model(source, model_type)
                        flux_points = get_flux_points_2hwc(source, model_type)
                        dict_fp[source.name] = True
                        self._create_flux_points_dataset(f"{label} ({model_type})", models, datasets,  spectral_model, flux_points)
                         
                elif tag == "1LHAASO":
                    for model_type in ['KM2A', 'WCDA']:
                        spectral_model = self._create_spectral_model(source, model_type)
                        flux_points = get_flux_points_1lhaaso(source, model_type)
                        dict_fp[source.name] = True
                        self._create_flux_points_dataset(f"{label} ({model_type})", models, datasets,  spectral_model, flux_points)
                
                elif tag == "vtscat":
                    unique_names = [source.name]
                    for flux_points in source.flux_points():
                        reference_id = flux_points.meta['reference_id']
                        model_name = generate_unique_name(source.name, reference_id, unique_names)
                        unique_names.append(model_name)
                        model = flux_points.reference_model
                        dict_fp[source.name] = True
                        spectral_model = model.spectral_model
                        label = model_name
                        self._create_flux_points_dataset(label, models, datasets,  spectral_model, flux_points)
                        
                else:        
                    spectral_model = source.spectral_model()
                    flux_points = source.flux_points
                    dict_fp[source.name] = True
                    self._create_flux_points_dataset(label, models, datasets,  spectral_model, flux_points)

                        
            except Exception as error:
                dict_fp[source.name] = False
                log.info(f"Unable to create dataset for {source.name}. Error: {error}")

        self.datasets = datasets
        self.datasets.models = models
        self._has_fp = dict_fp

    @staticmethod
    def _create_spectral_model(source, which=None):
        """Helper function to create a FluxPointsDataset for a source."""
        return source.spectral_model(which=which) if which else source.spectral_model()

    def _create_flux_points_dataset(self,  name, models, datasets,  spectral_model, flux_points):
        """Helper function to create a FluxPointsDataset for a source."""
        
        e_ref_min = self.config.energy_range.min
        e_ref_max = self.config.energy_range.max
        log.info("Getting FluxPointsDataset.")
        
        model = SkyModel(
        name=name,
        spectral_model=spectral_model,
        datasets_names=name
        )
        dataset = FluxPointsDataset(
        models=model,
        data=flux_points,
        name=name
        )
        
        if any([e_ref_min !=  None, e_ref_max !=  None]):
            dataset = cut_energy_flux_points_datasets(
            dataset, 
            e_ref_min, 
            e_ref_max
            ) 
                    
        models.append(model)
        datasets.append(dataset)
        log.info(f"Dataset created: {dataset}")

    def set_models(self, models, extend=True):
        """Set models on datasets.
        Adds `FoVBackgroundModel` if not present already

        Parameters
        ----------
        models : `~gammapy.modeling.models.Models` or str
            Models object or YAML models string
        extend : bool
            Extend the exiting models on the datasets or replace them.
        """
        if not self.datasets or len(self.datasets) == 0:
            raise RuntimeError("Missing datasets")

        log.info("Reading model.")
        if isinstance(models, str):
            models = Models.from_yaml(models)
        elif isinstance(models, Models):
            pass
        elif isinstance(models, DatasetModels) or isinstance(models, list):
            models = Models(models)
        else:
            raise TypeError(f"Invalid type: {models!r}")

        if extend:
            models.extend(self.datasets.models)

        self.datasets.models = models
        
        log.info(models)

    def read_models(self, path, extend=True):
        """Read models from YAML file.

        Parameters
        ----------
        path : str
        Path to the model file.
        extend : bool, optional
        Extend the exiting models on the datasets or replace them.
        Default is True.
        """
        path = make_path(path)
        models = Models.read(path)
        self.set_models(models, extend=extend)
        log.info(f"Models loaded from {path}.")
 
    def read_sources(self, path):
        """Read sources from YAML file.

        Parameters
        ----------
        path : str
        Path to the model file.
        extend : bool, optional
        Extend the exiting sources on the datasets or replace them.
        Default is True.
        """
        path = make_path(path)
        _sources = Sources()
        _sources.read(path)
        self.sources = Sources(_sources)
        log.info(f"sources loaded from {path}.")
    
    def write_sources(self, overwrite=True):
        """Write sources to YAML file.

        File name is taken from the configuration file.
        """
        filename_sources = self.config.general.sources_file
        if filename_sources is not None:
            self.sources.write(
                filename_sources, overwrite=overwrite)
            log.info(f"sources loaded from {filename_sources}.")
        else:
            raise RuntimeError("Missing sources_file in config.general")
            
    def write_models(self, overwrite=True, write_covariance=True):
        """Write models to YAML file.

        File name is taken from the configuration file.
        """
        filename_models = self.config.general.models_file
        if filename_models is not None:
            self.models.write(
                filename_models, overwrite=overwrite, write_covariance=write_covariance
            )
            log.info(f"Models loaded from {filename_models}.")
        else:
            raise RuntimeError("Missing models_file in config.general")

    def read_datasets(self):
        """Read datasets from YAML file.

        File names are taken from the configuration file.
        """
        filename = self.config.general.datasets_file
        filename_models = self.config.general.models_file
        if filename is not None:
            self.datasets = Datasets.read(filename)
            log.info(f"Datasets loaded from {filename}.")
            if filename_models is not None:
                self.read_models(filename_models, extend=False)
        else:
            raise RuntimeError("Missing datasets_file in config.general")

    def write_datasets(self, overwrite=True, write_covariance=True):
        """Write datasets to YAML file.

        File names are taken from the configuration file.

        Parameters
        ----------
        overwrite : bool, optional
            Overwrite existing file. Default is True.
        write_covariance : bool, optional
            Save covariance or not. Default is True.
        """
        filename = self.config.general.datasets_file
        filename_models = self.config.general.models_file
        if filename is not None:
            self.datasets.write(
                filename,
                filename_models,
                overwrite=overwrite,
                write_covariance=write_covariance,
            )
            log.info(f"Datasets stored to {filename}.")
            log.info(f"Datasets stored to {filename_models}.")
        else:
            raise RuntimeError("Missing datasets_file in config.general")